In [30]:
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()

bucket = "propelnoi-vijay-datalake"

train_s3 = f"s3://{bucket}/modeling/market_rent/train/"
validation_s3 = f"s3://{bucket}/modeling/market_rent/validation/"
output_s3 = f"s3://{bucket}/model-artifacts/market_rent/"

container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_s3,
    sagemaker_session=session
)

xgb.set_hyperparameters(
    objective="reg:squarederror",
    eval_metric="rmse",
    num_round=300,
    max_depth=5,
    eta=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1
)

train_input = TrainingInput(
    s3_data=train_s3,
    content_type="csv"
)

validation_input = TrainingInput(
    s3_data=validation_s3,
    content_type="csv"
)

xgb.fit(
    {
        "train": train_input,
        "validation": validation_input
    },
    logs=True
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-08-13-18-37-823


2026-03-08 13:18:38 Starting - Starting the training job...
2026-03-08 13:18:52 Starting - Preparing the instances for training...
2026-03-08 13:19:17 Downloading - Downloading input data...
2026-03-08 13:19:57 Downloading - Downloading the training image......
2026-03-08 13:21:08 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-08 13:21:12.223 ip-10-0-248-215.ap-south-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-08 13:21:12.288 ip-10-0-248-215.ap-south-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-08:13:21:12:INFO] Imported framework sag

In [31]:
# ============================================================
# Market-Rent Harmonizer - Stable Notebook Scoring Script
# ============================================================

import os
import tarfile
import boto3
import pandas as pd
import numpy as np
import xgboost as xgb

# -----------------------------
# 1. Configuration
# -----------------------------
BUCKET = "propelnoi-vijay-datalake"

MODEL_ARTIFACT_KEY = (
    "model-artifacts/market_rent/"
    "sagemaker-xgboost-2026-03-08-13-18-37-823/"
    "output/model.tar.gz"
)

INFERENCE_PREFIX = "modeling/market_rent/inference_input/"
FINAL_OUTPUT_KEY = "outputs/output_market_rent_predictions/output_market_rent_predictions.parquet"

MIN_RENT = 50.0
MAX_RENT = 10000.0
MIN_LOG_PRED = -10.0
MAX_LOG_PRED = 10.0

# New business controls
DEFAULT_INFLATION_RATE = 3.0
AT_MARKET_THRESHOLD = 0.15   # 15%
MAX_RECOMMENDED_MULTIPLIER = 1.50
MIN_RECOMMENDED_MULTIPLIER = 0.50

# -----------------------------
# 2. AWS clients
# -----------------------------
s3 = boto3.client("s3")

# -----------------------------
# 3. Helper functions
# -----------------------------
def list_s3_csv_files(bucket: str, prefix: str) -> list:
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return sorted(
        [
            obj["Key"]
            for obj in response.get("Contents", [])
            if obj["Key"].endswith(".csv")
        ]
    )

def download_s3_file(bucket: str, key: str, local_path: str) -> None:
    s3.download_file(bucket, key, local_path)

def find_model_file(extract_dir: str) -> str:
    files = os.listdir(extract_dir)
    if not files:
        raise FileNotFoundError("No files found after extracting model artifact.")

    for fname in files:
        if fname == "xgboost-model":
            return os.path.join(extract_dir, fname)

    for fname in files:
        if "xgboost" in fname.lower():
            return os.path.join(extract_dir, fname)

    return os.path.join(extract_dir, files[0])

# -----------------------------
# 4. Download and load model
# -----------------------------
local_model_tar = "/tmp/market_rent_model.tar.gz"
extract_dir = "/tmp/market_rent_model"
os.makedirs(extract_dir, exist_ok=True)

print("Downloading model artifact from S3...")
download_s3_file(BUCKET, MODEL_ARTIFACT_KEY, local_model_tar)

print("Extracting model artifact...")
with tarfile.open(local_model_tar) as tar:
    tar.extractall(path=extract_dir)

print("Extracted files:", os.listdir(extract_dir))

model_file = find_model_file(extract_dir)
print("Loading model from:", model_file)

model = xgb.Booster()
model.load_model(model_file)
print("Model loaded successfully.")

# -----------------------------
# 5. Download inference CSV
# -----------------------------
print("Locating inference CSV...")
csv_files = list_s3_csv_files(BUCKET, INFERENCE_PREFIX)
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in s3://{BUCKET}/{INFERENCE_PREFIX}")

inference_key = csv_files[-1]
print("Using inference CSV:", inference_key)

local_inference_csv = "/tmp/market_rent_inference.csv"
download_s3_file(BUCKET, inference_key, local_inference_csv)

print("Reading inference CSV...")
inference_df = pd.read_csv(local_inference_csv)
print("Inference shape:", inference_df.shape)
print(inference_df.head())

# -----------------------------
# 6. Validate required columns
# -----------------------------
meta_cols = [
    "property_id",
    "month",
    "city",
    "property_type",
    "units",
    "current_portfolio_rent"
]

feature_cols = [
    "city_idx",
    "room_type_idx",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "latitude",
    "longitude"
]

missing_meta = [c for c in meta_cols if c not in inference_df.columns]
missing_feat = [c for c in feature_cols if c not in inference_df.columns]

if missing_meta:
    raise ValueError(f"Missing metadata columns in inference CSV: {missing_meta}")

if missing_feat:
    raise ValueError(f"Missing feature columns in inference CSV: {missing_feat}")

# -----------------------------
# 7. Prepare metadata and features
# -----------------------------
meta_df = inference_df[meta_cols].copy()

if "inflation_rate" in inference_df.columns:
    meta_df["inflation_rate"] = pd.to_numeric(
        inference_df["inflation_rate"], errors="coerce"
    ).fillna(DEFAULT_INFLATION_RATE)
else:
    meta_df["inflation_rate"] = DEFAULT_INFLATION_RATE

numeric_cols = [
    "units",
    "current_portfolio_rent",
    "inflation_rate"
] + feature_cols

for col_name in numeric_cols:
    if col_name in inference_df.columns:
        inference_df[col_name] = pd.to_numeric(inference_df[col_name], errors="coerce")

meta_df["units"] = pd.to_numeric(meta_df["units"], errors="coerce")
meta_df["current_portfolio_rent"] = pd.to_numeric(meta_df["current_portfolio_rent"], errors="coerce")
meta_df["inflation_rate"] = pd.to_numeric(meta_df["inflation_rate"], errors="coerce").fillna(DEFAULT_INFLATION_RATE)

X = inference_df[feature_cols].copy()
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X = X.fillna(0.0)

print("Feature matrix shape:", X.shape)
print("Any NaN left in X:", X.isna().any().any())
print("Any inf left in X:", np.isinf(X.values).any())

# -----------------------------
# 8. Score predictions safely
# -----------------------------
print("Scoring XGBoost predictions...")
dmatrix = xgb.DMatrix(X)

preds_log = model.predict(dmatrix)
preds_log = np.array(preds_log, dtype=np.float64)
preds_log = np.nan_to_num(preds_log, nan=0.0, posinf=MAX_LOG_PRED, neginf=MIN_LOG_PRED)
preds_log = np.clip(preds_log, MIN_LOG_PRED, MAX_LOG_PRED)

preds = np.expm1(preds_log)
preds = np.nan_to_num(preds, nan=MIN_RENT, posinf=MAX_RENT, neginf=MIN_RENT)
preds = np.clip(preds, MIN_RENT, MAX_RENT)

meta_df["predicted_market_rent"] = preds

print("Prediction stats:")
print("min predicted_market_rent =", float(np.min(preds)))
print("max predicted_market_rent =", float(np.max(preds)))
print("mean predicted_market_rent =", float(np.mean(preds)))

# -----------------------------
# 9. Business logic
# -----------------------------
print("Calculating recommended_rent, rent_gap, rent_status, confidence_score...")

# base recommended rent
meta_df["recommended_rent_raw"] = meta_df["predicted_market_rent"] * (
    1 + meta_df["inflation_rate"] / 100.0
)

# cap recommendations relative to current rent to avoid extreme skew
meta_df["recommended_rent"] = np.where(
    meta_df["current_portfolio_rent"].notna(),
    np.minimum(
        np.maximum(
            meta_df["recommended_rent_raw"],
            meta_df["current_portfolio_rent"] * MIN_RECOMMENDED_MULTIPLIER
        ),
        meta_df["current_portfolio_rent"] * MAX_RECOMMENDED_MULTIPLIER
    ),
    meta_df["recommended_rent_raw"]
)

meta_df["rent_gap"] = meta_df["recommended_rent"] - meta_df["current_portfolio_rent"]

meta_df["rent_gap_pct"] = np.where(
    meta_df["current_portfolio_rent"].notna() & (meta_df["current_portfolio_rent"] != 0),
    meta_df["rent_gap"] / meta_df["current_portfolio_rent"],
    np.nan
)

def classify_rent_status(gap_pct: float) -> str:
    if pd.isna(gap_pct):
        return "UNKNOWN"
    if gap_pct > AT_MARKET_THRESHOLD:
        return "UNDER_RENTED"
    if gap_pct < -AT_MARKET_THRESHOLD:
        return "OVER_RENTED"
    return "AT_MARKET"

meta_df["rent_status"] = meta_df["rent_gap_pct"].apply(classify_rent_status)

meta_df["confidence_score"] = 0.80

meta_df["annual_rent_opportunity"] = np.where(
    meta_df["rent_status"] == "UNDER_RENTED",
    meta_df["rent_gap"] * 12,
    0.0
)

# -----------------------------
# 10. Final output
# -----------------------------
output_df = meta_df[
    [
        "property_id",
        "month",
        "city",
        "current_portfolio_rent",
        "predicted_market_rent",
        "recommended_rent",
        "rent_gap",
        "rent_gap_pct",
        "rent_status",
        "confidence_score",
        "annual_rent_opportunity"
    ]
].copy()

for c in [
    "current_portfolio_rent",
    "predicted_market_rent",
    "recommended_rent",
    "rent_gap",
    "rent_gap_pct",
    "confidence_score",
    "annual_rent_opportunity"
]:
    output_df[c] = pd.to_numeric(output_df[c], errors="coerce").round(4)

output_df["month"] = pd.to_datetime(output_df["month"], errors="coerce").dt.strftime("%Y-%m-%d")

print("Final output preview:")
display(output_df.head(10))

print("\nSummary stats:")
print("Rows:", len(output_df))
print("Cities:", output_df["city"].nunique())
print(output_df["rent_status"].value_counts(dropna=False))

print("\nRange checks:")
print(output_df[["predicted_market_rent", "current_portfolio_rent", "rent_gap_pct"]].describe())

# -----------------------------
# 11. Save parquet locally
# -----------------------------
local_output_parquet = "/tmp/output_market_rent_predictions.parquet"
output_df.to_parquet(local_output_parquet, index=False)
print("Saved local parquet:", local_output_parquet)

# -----------------------------
# 12. Upload parquet to S3
# -----------------------------
s3.upload_file(
    local_output_parquet,
    BUCKET,
    FINAL_OUTPUT_KEY
)

print(f"Uploaded final predictions to s3://{BUCKET}/{FINAL_OUTPUT_KEY}")

Extracting model artifact...
Extracted files: ['xgboost-model']
Loading model from: /tmp/market_rent_model/xgboost-model
Model loaded successfully.
Locating inference CSV...
Using inference CSV: modeling/market_rent/inference_input/part-00000-94c7f0a5-b4a8-4ff8-a23c-cff6fa49d6f8-c000.csv


/tmp/ipykernel_1046/2172417864.py:85: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Reading inference CSV...
Inference shape: (22140, 15)
  property_id       month     city property_type  units  \
0      P00001  2025-09-01  Seattle     Mixed Use   58.0   
1      P00001  2025-07-01  Seattle     Mixed Use   58.0   
2      P00001  2025-06-01  Seattle     Mixed Use   58.0   
3      P00001  2025-02-01  Seattle     Mixed Use   58.0   
4      P00001  2025-01-01  Seattle     Mixed Use   58.0   

   current_portfolio_rent  inflation_rate  city_idx  room_type_idx  \
0             2361.342813             3.0       3.0            1.0   
1             2411.855000             3.0       3.0            1.0   
2             2441.707500             3.0       3.0            1.0   
3             2406.944063             3.0       3.0            1.0   
4             2372.647500             3.0       3.0            1.0   

   minimum_nights  availability_365  number_of_reviews  review_scores_rating  \
0       10.990709        207.709405          82.307604              4.808836   
1       10

,property_id,month,city,current_portfolio_rent,predicted_market_rent,recommended_rent,rent_gap,rent_gap_pct,rent_status,confidence_score,annual_rent_opportunity
0,P00001,2025-09-01,Seattle,2361.3428,1653.9313,1703.5492,-657.7936,-0.2786,OVER_RENTED,0.8,0.0
1,P00001,2025-07-01,Seattle,2411.8550,1653.9313,1703.5492,-708.3058,-0.2937,OVER_RENTED,0.8,0.0
2,P00001,2025-06-01,Seattle,2441.7075,1653.9313,1703.5492,-738.1583,-0.3023,OVER_RENTED,0.8,0.0
3,P00001,2025-02-01,Seattle,2406.9441,1653.9313,1703.5492,-703.3949,-0.2922,OVER_RENTED,0.8,0.0
4,P00001,2025-01-01,Seattle,2372.6475,1653.9313,1703.5492,-669.0983,-0.2820,OVER_RENTED,0.8,0.0
5,P00001,2024-11-01,Seattle,2306.0069,1653.9313,1703.5492,-602.4577,-0.2613,OVER_RENTED,0.8,0.0
6,P00001,2023-10-01,Seattle,2228.4147,1653.9313,1703.5492,-524.8655,-0.2355,OVER_RENTED,0.8,0.0
7,P00001,2023-01-01,Seattle,2207.8891,1653.9313,1703.5492,-504.3399,-0.2284,OVER_RENTED,0.8,0.0
8,P00002,2025-06-01,Chicago,3006.4955,1601.0255,1649.0562,-1357.4392,-0.4515,OVER_RENTED,0.8,0.0
9,P00002,2025-04-01,Chicago,3024.5945,1601.0255,1649.0562,-1375.5383,-0.4548,OVER_RENTED,0.8,0.0



Summary stats:
Rows: 22140
Cities: 6
rent_status
AT_MARKET       11472
UNDER_RENTED     6721
OVER_RENTED      3947
Name: count, dtype: int64

Range checks:
       predicted_market_rent  current_portfolio_rent  rent_gap_pct
count           22140.000000            22140.000000  22140.000000
mean             2423.408350             2457.825353      0.038630
std               340.027854              377.936559      0.213492
min              1601.025500             1598.681600     -0.500000
25%              2421.317500             2177.897475     -0.095200
50%              2554.496000             2418.741550      0.045000
75%              2572.830100             2706.340775      0.181325
max              2733.247200             3892.544800      0.500000
Saved local parquet: /tmp/output_market_rent_predictions.parquet
Uploaded final predictions to s3://propelnoi-vijay-datalake/outputs/output_market_rent_predictions/output_market_rent_predictions.parquet
